[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navjotts/ML-experiments/blob/master/17%20-%20K-fold%20Cross%20Validation/K_fold_Cross_Validation.ipynb)

# Challenge

The challenge of training machine learning models is to be able to make accurate predictions on previously unseen real-world data in spite of the fact that we only have a finite training dataset to learn from. 

One way of validating our model's quality-of-fit and avoiding overfitting/underfitting, is to split the dataset into train and test (like the `train_test_split` method which sklearn provides). With this method, the randomly selected test dataset can be used to evaluate how our model performs on data that it has not yet seen in the training process. However, there are downsides to this approach:

*   We lose a valuable portion of data that we would prefer to be able to train on to serve as the test dataset. We would prefer to have both the testing and training datasets be as large as possible.
*   With small datasets, measures of our model's quality using the `train_test_split` method often have a high variance. (We can see this behavior below by changing the random seed when using `train_test_split`)

We can reduce the severity of both of these drawbacks by using what is called **K-fold Cross Validation**.


---


### Lets first see the problem with using something like `train_test_split`

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

titanic = sns.load_dataset('titanic')

# drop duplicate/analogous columns
titanic = titanic.drop(['alive',
                        'adult_male',
                        'sex',
                        'class',
                        'embark_town'], axis=1)

# take care of missing data
titanic['embarked'] = titanic['embarked'].fillna(method='ffill')
titanic = titanic.drop(['deck'], axis=1)
titanic['age'] = titanic['age'].fillna(method='ffill')

# convert binomials and categoricals to encoded labels
for label in ['embarked', 'who', 'alone']:
    titanic[label] = LabelEncoder().fit_transform(titanic[label])

titanic.head()

,survived,pclass,age,sibsp,parch,fare,embarked,who,alone
0,0,3,22.0,1,0,7.2500,2,1,0
1,1,1,38.0,1,0,71.2833,0,2,0
2,1,3,26.0,0,0,7.9250,2,2,1
3,1,1,35.0,1,0,53.1000,2,2,0
4,0,3,35.0,0,0,8.0500,2,1,1


In [0]:
from sklearn import linear_model
from sklearn import model_selection

# returns the accuracy
def run_Logistic_Regression(x, y, test_size_ratio, random_seed):
  x_train, x_test, y_train, y_test = model_selection.train_test_split(x, y, test_size=test_size_ratio, random_state=random_seed) 
  regr = linear_model.LogisticRegression()
  regr.fit(x_train, y_train)
  y_predict = regr.predict(x_test)
  error = y_predict-y_test
  accuracy = 1 - sum(error!=0)/len(error)
  
  return accuracy

In [3]:
labels = titanic['survived']
features = titanic.drop(['survived'], axis=1)

accuracies = []
test_size_ratio = 0.2
random_seeds = np.random.randint(low=1, high=100, size=5)
for seed in random_seeds:  
  accuracy = run_Logistic_Regression(features.as_matrix(), 
                                             labels.as_matrix(),
                                             test_size_ratio,
                                             seed)
  print("%d, %f" % (seed, accuracy))
  accuracies.append(accuracy)

print("\nAccuracies: ", accuracies)
print("Mean: ", np.mean(accuracies))
print("Std: ", np.std(accuracies))

75, 0.765363
16, 0.731844
43, 0.743017
51, 0.726257
74, 0.787709

Accuracies:  [0.7653631284916201, 0.7318435754189945, 0.7430167597765363, 0.7262569832402235, 0.7877094972067039]
Mean:  0.7508379888268155
Std:  0.022788914027230316


**Observation:**  volatile, high variane in accuracy rate – when we see the accuracy with the last seed **0.7877094972067039** – how can we claim anything when the previous seed gave **0.7262569832402235**

# K-fold Cross Validation (Let's build it)

The idea is to:

1.   Split the dataset into k-parts (each split is called a fold)
2.   The model is then trained on (k-1) folds with 1 fold held back
3.   The model's performance/accuracy is then tested on the held back fold
4.   Step 2 and 3 are repeated such that each fold is given a chance to be the held back test set
5.   After finishing the above, you end up with k different performance/accuracy scores  – which then you can summarize using a mean and a standard deviation

The result is a more reliable estimate of the performance of the model on new data – as it has been trained and evaluated multiple times on different data.

**One more advantage of Cross Validation is that it not only gives us an estimate of the performance of the model, but also a measure of how precise (i.e. standard deviation) this estimate is.**

(As a last step after the above, before throwing your model on new data, the model is trained on the entire dataset 1 final time – as we already have our test results. Hennce via this route, we don't have to sacrifice a valuable portion of our data for testing.)



In [0]:
def run_kfold_cv(data, target_feature, k):
  print("%d-fold Cross Validation" % k)
  # shuffle the dataset, so that we don't start with any prior bias
  data = data.reindex(np.random.permutation(data.index))

  total_size = len(data)
  accuracies = []
  for i in range(k):  
    start_index = i*int(total_size/k)
    end_index = start_index + int(total_size/k) - 1
    print("==Fold %d==" % (i+1))
    test_dataset = data.iloc[start_index:end_index, :]
    train_dataset = data.drop(test_dataset.index)

    y_train, x_train = train_dataset[target_feature].as_matrix(), train_dataset.drop([target_feature], axis=1).as_matrix()
    y_test, x_test = test_dataset[target_feature].as_matrix(), test_dataset.drop([target_feature], axis=1).as_matrix()

    regr = linear_model.LogisticRegression()
    regr.fit(x_train, y_train)
    y_predict = regr.predict(x_test)
    error = y_predict-y_test
    accuracy = 1 - sum(error!=0)/len(error)
    print("Accuracy: ", accuracy)

    accuracies.append(accuracy)

  print("\nAccuracies: ", accuracies)
  print("Mean: ", np.mean(accuracies))
  print("Std: ", np.std(accuracies))  

### Let's try with k=5

In [12]:
run_kfold_cv(titanic, 'survived', 5)

5-fold Cross Validation
==Fold 1==
Accuracy:  0.8022598870056497
==Fold 2==
Accuracy:  0.728813559322034
==Fold 3==
Accuracy:  0.7231638418079096
==Fold 4==
Accuracy:  0.7909604519774012
==Fold 5==
Accuracy:  0.6949152542372881

Accuracies:  [0.8022598870056497, 0.728813559322034, 0.7231638418079096, 0.7909604519774012, 0.6949152542372881]
Mean:  0.7480225988700566
Std:  0.041455223339760774


### Let's try with k=3

In [13]:
run_kfold_cv(titanic, 'survived', 3)

3-fold Cross Validation
==Fold 1==
Accuracy:  0.7263513513513513
==Fold 2==
Accuracy:  0.7533783783783784
==Fold 3==
Accuracy:  0.7533783783783784

Accuracies:  [0.7263513513513513, 0.7533783783783784, 0.7533783783783784]
Mean:  0.7443693693693695
Std:  0.012740662724081964


### Let's also try with k=10

In [14]:
run_kfold_cv(titanic, 'survived', 10)

10-fold Cross Validation
==Fold 1==
Accuracy:  0.8068181818181819
==Fold 2==
Accuracy:  0.7272727272727273
==Fold 3==
Accuracy:  0.7386363636363636
==Fold 4==
Accuracy:  0.8181818181818181
==Fold 5==
Accuracy:  0.75
==Fold 6==
Accuracy:  0.7613636363636364
==Fold 7==
Accuracy:  0.8181818181818181
==Fold 8==
Accuracy:  0.7272727272727273
==Fold 9==
Accuracy:  0.6590909090909092
==Fold 10==
Accuracy:  0.75

Accuracies:  [0.8068181818181819, 0.7272727272727273, 0.7386363636363636, 0.8181818181818181, 0.75, 0.7613636363636364, 0.8181818181818181, 0.7272727272727273, 0.6590909090909092, 0.75]
Mean:  0.7556818181818181
Std:  0.04664630852675859


**Observation:** The optimal number of folds is the one which allows for the test_size of each partition to be large enough to be a reasonable sample of the problem, whilst allowing enough iterations of the train_test evaluation to calculate a fair estimate of the accuracy.

**Example:** in the above case, 10 seems to be too large, and 3 seems too small a number – 5 feels optimal.

# Using sklearn

In [0]:
RANDOM_SEED = 42

def run_kfold_cv_using_sklearn(data, target_feature, k):
  kfold = model_selection.KFold(n_splits=k, random_state=RANDOM_SEED)
  model = linear_model.LogisticRegression()
  Y = data[target_feature]
  X = data.drop([target_feature], axis=1)
  results = model_selection.cross_val_score(model, X.as_matrix(), Y.as_matrix(), cv=kfold)
  
  print("Accuracies: ", results)
  print("Mean: ", np.mean(results))
  print("Std: ", np.std(results))    

In [16]:
run_kfold_cv_using_sklearn(titanic, 'survived', 5)

Accuracies:  [0.78212291 0.73595506 0.7247191  0.70786517 0.79213483]
Mean:  0.7485594124662608
Std:  0.032889441784782154


In [17]:
run_kfold_cv_using_sklearn(titanic, 'survived', 3)

Accuracies:  [0.73400673 0.72727273 0.76094276]
Mean:  0.7407407407407408
Std:  0.014547117168143331


In [18]:
run_kfold_cv_using_sklearn(titanic, 'survived', 10)

Accuracies:  [0.81111111 0.74157303 0.70786517 0.75280899 0.70786517 0.73033708
 0.7752809  0.65168539 0.79775281 0.7752809 ]
Mean:  0.7451560549313359
Std:  0.04554173132762068
